<a href="https://colab.research.google.com/github/rohit-jain8/Problem-Set-2/blob/main/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [2]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""

    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()

    had_support_ticket = rng.binomial(1, 0.22, size=n)

    s = (base + 0.35 * engagement_z
         - 0.60 * had_support_ticket
         + rng.normal(0, 0.75, size=n))

    satisfaction = np.clip(np.round(s), 1, 5)

    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction
    })

In [3]:
before = simulate_population(BASE_BEFORE)
after = simulate_population(BASE_AFTER)

print("Before true mean:", before["satisfaction"].mean())
print("After true mean:", after["satisfaction"].mean())
print("True improvement:", after["satisfaction"].mean() - before["satisfaction"].mean())

before.head()

Before true mean: 3.29012
After true mean: 3.38602
True improvement: 0.09589999999999987


,engagement_minutes,engagement_z,had_support_ticket,s,satisfaction
0,39.759,0.003,0,2.888,3.0
1,17.743,-0.833,0,3.016,3.0
2,51.949,0.466,0,2.948,3.0
3,58.226,0.705,1,3.255,3.0
4,10.272,-1.117,0,3.433,3.0


I simulated 50,000 users before and after the onboarding redesign using the specified parameters. Satisfaction depends on user engagement, whether a user submitted a support ticket, and random variation. Higher engagement increases satisfaction, while having a support ticket decreases it. I then rounded and restricted satisfaction scores to the 1–5 scale.

The true population mean satisfaction was 3.29012 before the redesign and 3.38602 after, representing an improvement of approximately 0.096 points. These results are close to the expected values of 3.30 and 3.40, confirming that the simulation captures the intended small improvement in satisfaction.

### Step 1.2: Impose the response mechanism (15 pts)

In [4]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""

    z = (a + b * (df["s"] - 3.3)
         + 0.30 * df["engagement_z"]
         - 0.25 * df["had_support_ticket"])

    p_respond = 1 / (1 + np.exp(-z))

    responded = rng.uniform(0, 1, size=len(df)) < p_respond

    return responded

In [5]:
before["responded"] = respond(before, a=-1.0, b=0.45)
after["responded"] = respond(after, a=-2.75, b=2.00)

before_observed = before.loc[before["responded"], "satisfaction"]
after_observed = after.loc[after["responded"], "satisfaction"]

print("Before observed mean:", before_observed.mean())
print("After observed mean:", after_observed.mean())
print("Observed improvement:", after_observed.mean() - before_observed.mean())

print("Before response rate:", before["responded"].mean())
print("After response rate:", after["responded"].mean())

Before observed mean: 3.5999418942475305
After observed mean: 4.304067395024352
Observed improvement: 0.7041255007768212
Before response rate: 0.27536
After response rate: 0.15194


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

The survey suffers from nonresponse bias because the probability of answering depends on a user's satisfaction. More satisfied users are more likely to respond, particularly after the redesign, which causes the observed survey averages to overstate satisfaction in the overall population.

The observed satisfaction increased from 3.59994 before the redesign to 4.30407 afterward, suggesting an improvement of approximately 0.704 points. However, the true population improvement was only about 0.096 points. Additionally, the response rate fell from 27.536% before the redesign to 15.194% afterward, meaning the survey reporting higher satisfaction actually represents a smaller share of users.

Increasing the sample size would not eliminate this bias because the responses are not randomly selected from the population. A larger sample would reduce sampling variability, but the estimated satisfaction could remain systematically higher than the true population mean.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [6]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""

    x = np.asarray(x)
    results = []

    for i in range(B):
        sample = rng.choice(x, size=len(x), replace=True)
        results.append(stat(sample))

    return np.array(results)

In [7]:
x = after_observed.to_numpy()

boot_means = bootstrap_by_hand(x, np.mean)
boot_medians = bootstrap_by_hand(x, np.median)

mean_ci_manual = np.percentile(boot_means, [2.5, 97.5])
median_ci_manual = np.percentile(boot_medians, [2.5, 97.5])

print("Observed mean:", np.mean(x))
print("Manual mean percentile CI:", mean_ci_manual)

print("Observed median:", np.median(x))
print("Manual median percentile CI:", median_ci_manual)

Observed mean: 4.304067395024352
Manual mean percentile CI: [4.2891931  4.31920495]
Observed median: 4.0
Manual median percentile CI: [4. 4.]


In [8]:
from scipy import stats

scipy_mean_percentile = stats.bootstrap(
    (x,), np.mean, n_resamples=2000,
    confidence_level=0.95, method="percentile",
    random_state=42
)

scipy_mean_bca = stats.bootstrap(
    (x,), np.mean, n_resamples=2000,
    confidence_level=0.95, method="BCa",
    random_state=42
)

scipy_median_percentile = stats.bootstrap(
    (x,), np.median, n_resamples=2000,
    confidence_level=0.95, method="percentile",
    random_state=42
)

scipy_median_bca = stats.bootstrap(
    (x,), np.median, n_resamples=2000,
    confidence_level=0.95, method="BCa",
    random_state=42
)

print("SciPy mean percentile CI:", scipy_mean_percentile.confidence_interval)
print("SciPy mean BCa CI:", scipy_mean_bca.confidence_interval)

print("SciPy median percentile CI:", scipy_median_percentile.confidence_interval)
print("SciPy median BCa CI:", scipy_median_bca.confidence_interval)

SciPy mean percentile CI: ConfidenceInterval(low=np.float64(4.2885316572331185), high=np.float64(4.319468211135975))
SciPy mean BCa CI: ConfidenceInterval(low=np.float64(4.288580790426669), high=np.float64(4.319731472949849))
SciPy median percentile CI: ConfidenceInterval(low=np.float64(4.0), high=np.float64(4.0))
SciPy median BCa CI: ConfidenceInterval(low=np.float64(nan), high=np.float64(nan))


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


In [9]:
print("Manual mean CI:", mean_ci_manual)
print("SciPy mean CI:", scipy_mean_percentile.confidence_interval)

print("Lower endpoint difference:",
      abs(mean_ci_manual[0] - scipy_mean_percentile.confidence_interval.low))

print("Upper endpoint difference:",
      abs(mean_ci_manual[1] - scipy_mean_percentile.confidence_interval.high))

Manual mean CI: [4.2891931  4.31920495]
SciPy mean CI: ConfidenceInterval(low=np.float64(4.2885316572331185), high=np.float64(4.319468211135975))
Lower endpoint difference: 0.0006614453073581217
Upper endpoint difference: 0.00026326181387403835


I used 2,000 bootstrap resamples to estimate 95% confidence intervals for the post-redesign survey mean and median. The observed mean satisfaction was 4.3041, with a manually calculated percentile confidence interval of [4.2892, 4.3192]. SciPy produced a similar percentile interval of [4.2885, 4.3195] and a BCa interval of [4.2886, 4.3197]. The manual and SciPy percentile intervals differed by only 0.00066 at the lower endpoint and 0.00026 at the upper endpoint, confirming that both methods produced nearly identical results. The BCa adjustment also made very little difference for the mean.

The observed median was 4.0, and both the manual and SciPy percentile intervals were [4.0, 4.0]. However, SciPy could not calculate the median BCa interval because the jackknife estimates used for the acceleration adjustment were degenerate, making the adjustment undefined. This can occur with discrete satisfaction scores when removing individual observations does not change the median.

Although the confidence interval for the mean is narrow, it only captures uncertainty from resampling the survey respondents. It does not correct for nonresponse bias, meaning the true population mean of 3.3860 remains far outside the interval. Therefore, a precise estimate from a biased sample does not necessarily provide an accurate representation of the overall population.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [10]:
# Step 2.2: Coverage simulation

from scipy import stats

rng_coverage = np.random.default_rng(123)

true_mean = after["satisfaction"].mean()

R = 200
B = 500
sample_size = 200

percentile_covered = []
bca_covered = []

population = after["satisfaction"].to_numpy()

for i in range(R):
    sample = rng_coverage.choice(population, size=sample_size, replace=True)

    percentile_ci = stats.bootstrap(
        (sample,), np.mean,
        n_resamples=B,
        confidence_level=0.95,
        method="percentile",
        random_state=rng_coverage
    ).confidence_interval

    bca_ci = stats.bootstrap(
        (sample,), np.mean,
        n_resamples=B,
        confidence_level=0.95,
        method="BCa",
        random_state=rng_coverage
    ).confidence_interval

    percentile_covered.append(percentile_ci.low <= true_mean <= percentile_ci.high)
    bca_covered.append(bca_ci.low <= true_mean <= bca_ci.high)

percentile_coverage = np.mean(percentile_covered)
bca_coverage = np.mean(bca_covered)

percentile_mcse = np.sqrt(percentile_coverage * (1 - percentile_coverage) / R)
bca_mcse = np.sqrt(bca_coverage * (1 - bca_coverage) / R)

print("True population mean:", true_mean)
print("Percentile coverage:", percentile_coverage)
print("BCa coverage:", bca_coverage)
print("Percentile Monte Carlo SE:", percentile_mcse)
print("BCa Monte Carlo SE:", bca_mcse)
print("Coverage difference:", bca_coverage - percentile_coverage)

True population mean: 3.38602
Percentile coverage: 0.96
BCa coverage: 0.93
Percentile Monte Carlo SE: 0.013856406460551024
BCa Monte Carlo SE: 0.01804161855266871
Coverage difference: -0.029999999999999916


In [12]:
# Coverage when sampling from biased survey respondents

biased_population = after_observed.to_numpy()

biased_percentile_covered = []
biased_bca_covered = []

for i in range(R):
    sample = rng_coverage.choice(biased_population, size=sample_size, replace=True)

    percentile_ci = stats.bootstrap(
        (sample,), np.mean,
        n_resamples=B,
        confidence_level=0.95,
        method="percentile",
        random_state=rng_coverage
    ).confidence_interval

    bca_ci = stats.bootstrap(
        (sample,), np.mean,
        n_resamples=B,
        confidence_level=0.95,
        method="BCa",
        random_state=rng_coverage
    ).confidence_interval

    biased_percentile_covered.append(
        percentile_ci.low <= true_mean <= percentile_ci.high
    )

    biased_bca_covered.append(
        bca_ci.low <= true_mean <= bca_ci.high
    )

biased_percentile_coverage = np.mean(biased_percentile_covered)
biased_bca_coverage = np.mean(biased_bca_covered)

biased_percentile_mcse = np.sqrt(
    biased_percentile_coverage * (1 - biased_percentile_coverage) / R
)

biased_bca_mcse = np.sqrt(
    biased_bca_coverage * (1 - biased_bca_coverage) / R
)

print("Biased percentile coverage:", biased_percentile_coverage)
print("Biased BCa coverage:", biased_bca_coverage)
print("Biased percentile Monte Carlo SE:", biased_percentile_mcse)
print("Biased BCa Monte Carlo SE:", biased_bca_mcse)

coverage_differences = np.array(bca_covered, dtype=int) - np.array(percentile_covered, dtype=int)

difference_mcse = coverage_differences.std(ddof=1) / np.sqrt(R)

print("Coverage difference:", bca_coverage - percentile_coverage)
print("Monte Carlo SE of difference:", difference_mcse)

Biased percentile coverage: 0.0
Biased BCa coverage: 0.0
Biased percentile Monte Carlo SE: 0.0
Biased BCa Monte Carlo SE: 0.0
Coverage difference: -0.029999999999999916
Monte Carlo SE of difference: 0.014017218413804555


I ran 200 repeated sampling simulations to evaluate how often the 95% bootstrap confidence intervals contained the true population mean of 3.38602. When sampling randomly from the full population, the percentile interval achieved 96% coverage with a Monte Carlo standard error of 0.0139, while the BCa interval achieved 93% coverage with a Monte Carlo standard error of 0.0180. Both coverage rates were reasonably close to the expected 95%. The percentile method had a 3-percentage-point higher coverage rate, with a Monte Carlo standard error of approximately 1.40 percentage points for the difference. The percentile method had a 3-percentage-point higher coverage rate, compared to a Monte Carlo standard error of approximately 1.40 percentage points for the difference. While this suggests some variation between the methods in our simulation, it does not necessarily mean that the percentile method performs better in general.

I then repeated the simulation using the biased post-redesign survey respondents. Neither method covered the true population mean in any of the 200 simulations, resulting in estimated coverage rates of 0% for both methods. The calculated Monte Carlo standard errors were also zero because no intervals contained the true mean, although this does not establish that the underlying coverage probabilities are exactly zero. This occurs because the biased samples are centered around the observed respondent mean of approximately 4.304, rather than the true population mean of 3.386.

These results demonstrate that bootstrap confidence intervals can perform well when samples are representative but fail to capture the population truth when nonresponse bias is present. Increasing statistical precision does not eliminate systematic bias, meaning even a narrow confidence interval can consistently miss the true population mean.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [13]:
from scipy import stats
import numpy as np

# Survey responses before and after the redesign
before_scores = before_observed.to_numpy()
after_scores = after_observed.to_numpy()

# Welch's two-sample t-test
t_stat, p_value = stats.ttest_ind(
    after_scores,
    before_scores,
    equal_var=False
)

# Difference in observed means
mean_difference = np.mean(after_scores) - np.mean(before_scores)

# Cohen's d using pooled standard deviation
n_before = len(before_scores)
n_after = len(after_scores)

sd_before = np.std(before_scores, ddof=1)
sd_after = np.std(after_scores, ddof=1)

pooled_sd = np.sqrt(
    ((n_before - 1) * sd_before**2 +
     (n_after - 1) * sd_after**2) /
    (n_before + n_after - 2)
)

cohens_d = mean_difference / pooled_sd

print("Before observed mean:", np.mean(before_scores))
print("After observed mean:", np.mean(after_scores))
print("Observed mean difference:", mean_difference)
print("t-statistic:", t_stat)
print("p-value:", p_value)
print("Cohen's d:", cohens_d)
print("Before sample size:", n_before)
print("After sample size:", n_after)

Before observed mean: 3.5999418942475305
After observed mean: 4.304067395024352
Observed mean difference: 0.7041255007768212
t-statistic: 65.16599829587112
p-value: 0.0
Cohen's d: 0.8637067597399851
Before sample size: 13768
After sample size: 7597


I conducted Welch's two-sample t-test to compare average satisfaction among survey respondents before and after the onboarding redesign. The observed mean increased from 3.5999 before the redesign to 4.3041 afterward, representing an improvement of approximately 0.704 points. The test produced a t-statistic of 65.166 and a p-value below 0.001, indicating a statistically significant difference between the two respondent groups. Cohen's d was approximately 0.864, suggesting a large observed effect size.

However, statistical significance does not necessarily mean the redesign caused such a large improvement in satisfaction. Our simulation showed that the true population mean increased by only about 0.096 points, compared to the observed improvement of 0.704 points. This difference occurs because survey participation depends on satisfaction, particularly after the redesign, when more satisfied users were much more likely to respond. As a result, the post-redesign survey overrepresents satisfied users and exaggerates the apparent improvement.

Therefore, while the hypothesis test provides strong evidence that the observed respondent means differ, it does not establish that the redesign improved satisfaction by 0.704 points across the entire population. The main issue is nonresponse bias rather than statistical significance, and a very small p-value cannot correct for an unrepresentative sample.

### Step 3.2: Power, before the fact

In [15]:
from statsmodels.stats.power import TTestIndPower

# Standard deviation of satisfaction in the full populations
sd_before_true = before["satisfaction"].std(ddof=1)
sd_after_true = after["satisfaction"].std(ddof=1)

sd_planned = np.sqrt(
    (sd_before_true**2 + sd_after_true**2) / 2
)

# Planned true improvement of 0.10 points
true_effect = 0.10
d_planned = true_effect / sd_planned

# Sample size needed for 80% power
power_analysis = TTestIndPower()

n_required = power_analysis.solve_power(
    effect_size=d_planned,
    alpha=0.05,
    power=0.80
)

print("Planned Cohen's d:", d_planned)
print("Required sample size per group:", np.ceil(n_required))
print("Total required sample size:", 2 * np.ceil(n_required))

# Minimum detectable effect with the available responses
n_before = len(before_observed)
n_after = len(after_observed)

d_detectable = power_analysis.solve_power(
    nobs1=n_before,
    ratio=n_after / n_before,
    alpha=0.05,
    power=0.80
)

points_detectable = d_detectable * sd_planned

print("Available pre-redesign responses:", n_before)
print("Available post-redesign responses:", n_after)
print("Minimum detectable Cohen's d:", d_detectable)
print("Minimum detectable improvement in points:", points_detectable)

# Compare estimates using different sample sizes
sample_sizes = [100, 500, 1000, 3000, 5000]
repetitions = 200

rng_power = np.random.default_rng(123)

true_after_mean = after["satisfaction"].mean()
biased_responses = after_observed.to_numpy()

print("True post-redesign mean:", true_after_mean)

for n in sample_sizes:
    estimates = []

    for i in range(repetitions):
        sample = rng_power.choice(
            biased_responses,
            size=n,
            replace=True
        )

        estimates.append(np.mean(sample))

    average_estimate = np.mean(estimates)
    bias = average_estimate - true_after_mean
    variability = np.std(estimates, ddof=1)

    print("Sample size:", n)
    print("Average estimate:", round(average_estimate, 3))
    print("Bias:", round(bias, 3))
    print("Standard deviation of estimates:", round(variability, 3))
    print()

Planned Cohen's d: 0.11275059219137082
Required sample size per group: 1236.0
Total required sample size: 2472.0
Available pre-redesign responses: 13768
Available post-redesign responses: 7597
Minimum detectable Cohen's d: 0.04004003100780494
Minimum detectable improvement in points: 0.03551203610518095
True post-redesign mean: 3.38602
Sample size: 100
Average estimate: 4.303
Bias: 0.917
Standard deviation of estimates: 0.07

Sample size: 500
Average estimate: 4.306
Bias: 0.92
Standard deviation of estimates: 0.032

Sample size: 1000
Average estimate: 4.305
Bias: 0.919
Standard deviation of estimates: 0.023

Sample size: 3000
Average estimate: 4.305
Bias: 0.919
Standard deviation of estimates: 0.013

Sample size: 5000
Average estimate: 4.304
Bias: 0.918
Standard deviation of estimates: 0.009



For a true improvement of 0.10 satisfaction points, the planned standardized effect size is approximately 0.113. Using a 5% significance level and 80% statistical power, we would need approximately 1,236 respondents per group, or 2,472 total. With our actual sample sizes of 13,768 pre-redesign respondents and 7,597 post-redesign respondents, the minimum detectable improvement is approximately 0.036 points. These calculations assume independent, representative samples with approximately equal variances.

However, increasing the sample size does not necessarily improve the accuracy of our survey results because the response mechanism is biased. In our simulation, increasing the sample size from 100 to 5,000 reduced the standard deviation of our estimates from 0.070 to 0.009, but the bias remained approximately 0.92 points. The average estimated satisfaction stayed around 4.30, compared to the true population mean of 3.386.

This demonstrates that larger samples can reduce sampling variability without correcting systematic nonresponse bias. Our simulation assumes that the underlying response pattern remains unchanged as the sample size increases. Since users with higher satisfaction scores are more likely to respond, collecting more responses through the same mechanism would give us a more precise estimate of the wrong population mean. To improve accuracy, we would need to address the response mechanism rather than simply increase the sample size.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [18]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# Variables used to predict survey responses
columns = ["engagement_minutes", "had_support_ticket"]

# Fit the response model using the full population
X = after[columns]
y = after["responded"].astype(int)

response_model = LogisticRegression(max_iter=1000)
response_model.fit(X, y)

# Predict each user's probability of responding
predicted_response = response_model.predict_proba(X)[:, 1]

# Calculate weights for users who responded
respondents = after[after["responded"]].copy()
respondents["weight"] = 1 / predicted_response[y == 1]

# Calculate the original weighted estimate
weighted_mean_v2 = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

# Bootstrap the full population and refit the model
rng_v2 = np.random.default_rng(123)
bootstrap_estimates_v2 = []

for i in range(1000):

    # Resample all users, including nonrespondents
    sample_indices = rng_v2.choice(
        len(after),
        size=len(after),
        replace=True
    )

    sample = after.iloc[sample_indices].copy()

    # Refit logistic regression in each bootstrap sample
    X_sample = sample[columns]
    y_sample = sample["responded"].astype(int)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_sample, y_sample)

    # Recalculate response probabilities
    probabilities = model.predict_proba(X_sample)[:, 1]

    # Recalculate weights for respondents
    sample_respondents = sample[y_sample == 1].copy()
    sample_respondents["weight"] = 1 / probabilities[y_sample == 1]

    # Calculate the weighted mean for this bootstrap sample
    estimate = np.average(
        sample_respondents["satisfaction"],
        weights=sample_respondents["weight"]
    )

    bootstrap_estimates_v2.append(estimate)

# Calculate the 95% percentile confidence interval
weighted_ci_v2 = np.percentile(
    bootstrap_estimates_v2,
    [2.5, 97.5]
)

# Compare results with the known population mean
true_mean = after["satisfaction"].mean()
observed_mean = after_observed.mean()

print("Observed post-redesign mean:", observed_mean)
print("Revised weighted mean:", weighted_mean_v2)
print("True population mean:", true_mean)
print("Remaining bias:", weighted_mean_v2 - true_mean)
print("Revised 95% bootstrap confidence interval:", weighted_ci_v2)

Observed post-redesign mean: 4.304067395024352
Revised weighted mean: 4.143177046830373
True population mean: 3.38602
Remaining bias: 0.7571570468303732
Revised 95% bootstrap confidence interval: [4.12240155 4.16231029]


### Version 2 Results and Interpretation

The revised inverse-probability weighting model gave us a weighted satisfaction mean of **4.1432**, compared to the observed mean of **4.3041** and the true population mean of **3.3860**. While weighting helped reduce the bias from about 0.9180 to 0.7572 points, there was still a pretty significant difference between our estimate and the actual population mean.

The updated 95% bootstrap confidence interval was **[4.1224, 4.1623]**, compared to **[4.1224, 4.1634]** in Version 1. These intervals are almost identical, which suggests that accounting for uncertainty in the logistic regression model didn't make much of a difference in this case.

However, neither confidence interval includes the true population mean of 3.3860. This is important because it shows that even though we improved the way we calculated our confidence interval, we still haven't fixed the underlying nonresponse bias. Since users are more likely to respond based on their satisfaction, even after accounting for engagement and support-ticket history, the conditional independence assumption needed for inverse-probability weighting does not hold.

Overall, Version 2 gives us a better way to account for uncertainty, but it doesn't actually solve the main issue of selection bias in our survey.

### Task 4.1: AI Reflection and Verification

**Prompt Revision and Critique**

In Version 1, I noticed that the AI-generated bootstrap only resampled survey respondents while keeping their estimated response probabilities and weights fixed. This was a limitation because it didn't account for uncertainty from estimating the logistic regression model itself. For Version 2, I revised my prompt to have the AI resample the full simulated population, refit the logistic regression model, and recalculate the weights during each bootstrap repetition.

**Verification Against the True Population Mean**

After running Version 2, the weighted satisfaction mean was **4.1432**, compared to the true population mean of **3.3860**. This left a remaining bias of approximately **0.7572 points**. The revised 95% bootstrap confidence interval was **[4.1224, 4.1623]**, which still did not contain the true mean. Although weighting reduced some of the original bias, it did not fully correct the problem because response probability depended directly on satisfaction.

**How the Weights Work**

The logistic regression model estimates each user's probability of responding based on their engagement minutes and support-ticket history. For users who responded, the code calculates their weight by dividing 1 by their predicted response probability, meaning users who were less likely to respond receive more weight. The weighted mean then uses these weights to calculate an adjusted satisfaction average, giving more influence to respondents who represent users with lower response probabilities.

**My Judgment on Using AI**

I would use AI for a similar task again because it helped me implement and improve the weighting method, but I would still verify the results and check the underlying assumptions before trusting its conclusions.

### Task 4.2: The slide (5 pts)

**Post-Redesign Survey Results and Limitations**

Our post-redesign survey suggests that users who responded were generally satisfied with the updated product. However, this doesn't necessarily mean that the entire user population feels the same way, especially since users with higher satisfaction scores were more likely to respond.

The observed post-redesign satisfaction mean was **4.3041**, but after applying inverse-probability weighting, our estimate decreased to **4.1432**, with a 95% bootstrap confidence interval of **[4.1224, 4.1623]**. Although weighting helped reduce some of the bias, it wasn't enough to fully correct the issue because users' likelihood of responding still depended directly on their satisfaction.

Because of this, we can't confidently say that overall satisfaction improved as a result of the redesign. Our survey gives us an idea of how respondents felt, but it doesn't necessarily reflect the experience of all users or prove that the redesign caused an improvement.

Moving forward, I would recommend randomly selecting users to receive the survey and following up with those who don't respond. This would help reduce selection bias and give us a more accurate picture of how satisfied our overall user population actually is.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* GPT-6; October 20, 2026

*Version 1 prompt:*

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

*Reply:*

Task 4.1 — Version 1: Inverse-Probability Weighting
Python code
The following code uses logistic regression to estimate each user's probability of responding based on engagement_minutes and had_support_ticket. It then uses inverse-probability weighting (IPW) to estimate post-redesign satisfaction.

from sklearn.linear_model import LogisticRegression
import numpy as np

# Predict survey response using the two covariates
X = after[["engagement_minutes", "had_support_ticket"]]
y = after["responded"].astype(int)

response_model = LogisticRegression(max_iter=1000)
response_model.fit(X, y)

# Estimate the probability of responding
after["predicted_response"] = response_model.predict_proba(X)[:, 1]

# Calculate inverse-probability weights for respondents
respondents = after[after["responded"]].copy()
respondents["weight"] = 1 / respondents["predicted_response"]

# Calculate the weighted satisfaction mean
weighted_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

print("Observed post-redesign mean:", after_observed.mean())
print("Weighted post-redesign mean:", weighted_mean)
print("True post-redesign mean:", after["satisfaction"].mean())
print("Remaining bias:", weighted_mean - after["satisfaction"].mean())

# Bootstrap confidence interval for the weighted mean
weighted_estimates = []

for i in range(1000):
    sample = respondents.sample(
        n=len(respondents),
        replace=True,
        random_state=i
    )

    estimate = np.average(
        sample["satisfaction"],
        weights=sample["weight"]
    )

    weighted_estimates.append(estimate)

weighted_ci = np.percentile(weighted_estimates, [2.5, 97.5])

print("95% bootstrap confidence interval:", weighted_ci)

How the weighting works
The logistic regression estimates the probability that each user responds based on their engagement and support-ticket history.

We then calculate the weight for each respondent as:
\[
w_i=\frac{1}{\hat{P}(\text{respond}_i=1\mid X_i)}
\]

For example, someone with a predicted response probability of 0.20 receives a weight of 5, while someone with a predicted probability of 0.50 receives a weight of 2.

This gives more influence to respondents who represent users who were less likely to participate.

The weighted mean is calculated as:
\[
\bar{s}_{IPW}=\frac{\sum_{i:R_i=1}w_i s_i}{\sum_{i:R_i=1}w_i}
\]

This is a normalized inverse-probability-weighted estimate of satisfaction.

The bootstrap then repeatedly resamples respondents and recalculates the weighted mean to produce a 95% percentile confidence interval.

One limitation is that the bootstrap keeps the estimated weights fixed rather than refitting the response model in every resample. Consequently, it does not fully account for uncertainty in the estimated response probabilities.
Mechanism check: What assumption makes this correction valid?
For inverse-probability weighting based on the two observed covariates to recover the population mean, we need conditional independence of response and satisfaction.

Specifically, after accounting for engagement minutes and support-ticket history, whether someone responds must be independent of their satisfaction score:
\[
R\perp S\mid X
\]

We also need every relevant type of user to have a positive probability of responding, and our model must estimate those response probabilities appropriately.

However, your simulated response mechanism explicitly makes response probability depend on satisfaction itself.

This means two users with identical engagement and support-ticket histories can still have different response probabilities because one is more satisfied than the other.

Therefore, the conditional independence assumption is violated.
Evaluation: Will this correction recover the true mean?
No.

The weighting procedure can adjust for differences associated with engagement minutes and support-ticket history, but it cannot fully correct satisfaction-based nonresponse.

Since more satisfied users are disproportionately likely to respond even after controlling for the observed covariates, the weighted sample will generally continue to overrepresent satisfied users.

Using the results from your earlier execution of this code:
Measure	Result
Unweighted observed mean	4.3041
IPW-adjusted mean	4.1432
Known true population mean	3.3860
Remaining bias	+0.7572
95% bootstrap interval	[4.1224, 4.1634]
The weighting reduces the overestimate but does not recover the true population mean. The confidence interval also excludes the known truth, illustrating how an estimate can appear statistically precise while remaining systematically biased.

The exact numerical results should be verified by running the code in your notebook.

*Version 2 prompt:*

[Prep] Act as a survey statistician who works on non-response adjustment for product research.

[Request] I used AI to create an inverse-probability weighting model for my simulated post-redesign satisfaction survey. The model used logistic regression with engagement_minutes and had_support_ticket to predict whether users would respond.

After running the Version 1 code, I got the following results:

- Observed post-redesign mean: 4.3041
- Weighted post-redesign mean: 4.1432
- True population mean: 3.38602
- Remaining bias: 0.7572
- 95% bootstrap confidence interval: [4.1224, 4.1634]

[Iterate] One issue I noticed with the original code is that the bootstrap only resamples respondents and keeps their estimated response probabilities and weights the same. This means we aren't accounting for uncertainty in the logistic regression model itself.

Can you revise the code so that each bootstrap sample includes the full simulated population, refits the logistic regression model, and recalculates the weights before estimating the weighted satisfaction mean? Please use 1,000 bootstrap repetitions and report the updated 95% confidence interval.

Keep the Python code relatively simple and consistent with what we've learned in class. Use numpy, pandas, and scikit-learn, and keep my existing variable names: after, satisfaction, engagement_minutes, had_support_ticket, and responded.

[Mechanism Check] Please explain what assumptions need to hold for inverse-probability weighting to work. Specifically, I want to understand whether weighting can actually correct the bias when users' likelihood of responding depends directly on their satisfaction, even after accounting for the other two variables.

[Evaluate] Compare the updated weighted estimate and confidence interval to the true population mean of 3.38602. Did the revised approach actually correct the bias, or did it just improve how we measure uncertainty? Please explain why.

*Reply:*

Version 2: Revised Inverse-Probability Weighting
The main issue with Version 1 is that the bootstrap only resampled respondents while keeping their estimated response probabilities and weights fixed. This means the confidence interval did not account for uncertainty from fitting the logistic regression model.

In Version 2, we'll improve this by resampling the full simulated population, refitting the logistic regression model, and recalculating the weights in each of the 1,000 bootstrap repetitions.

This improves our uncertainty estimation, although it does not necessarily correct the underlying nonresponse bias.
1. Revised Python code
The following code uses your existing after DataFrame and the same variables as Version 1.

from sklearn.linear_model import LogisticRegression
import numpy as np

# Variables used to predict survey responses
columns = ["engagement_minutes", "had_support_ticket"]

# Fit the response model using the full population
X = after[columns]
y = after["responded"].astype(int)

response_model = LogisticRegression(max_iter=1000)
response_model.fit(X, y)

# Predict each user's probability of responding
predicted_response = response_model.predict_proba(X)[:, 1]

# Calculate weights for users who responded
respondents = after[after["responded"]].copy()
respondents["weight"] = 1 / predicted_response[y == 1]

# Calculate the original weighted estimate
weighted_mean_v2 = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

# Bootstrap the full population and refit the model
rng_v2 = np.random.default_rng(123)
bootstrap_estimates_v2 = []

for i in range(1000):

    # Resample all users, including nonrespondents
    sample_indices = rng_v2.choice(
        len(after),
        size=len(after),
        replace=True
    )

    sample = after.iloc[sample_indices].copy()

    # Refit logistic regression in each bootstrap sample
    X_sample = sample[columns]
    y_sample = sample["responded"].astype(int)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_sample, y_sample)

    # Recalculate response probabilities
    probabilities = model.predict_proba(X_sample)[:, 1]

    # Recalculate weights for respondents
    sample_respondents = sample[y_sample == 1].copy()
    sample_respondents["weight"] = 1 / probabilities[y_sample == 1]

    # Calculate the weighted mean for this bootstrap sample
    estimate = np.average(
        sample_respondents["satisfaction"],
        weights=sample_respondents["weight"]
    )

    bootstrap_estimates_v2.append(estimate)

# Calculate the 95% percentile confidence interval
weighted_ci_v2 = np.percentile(
    bootstrap_estimates_v2,
    [2.5, 97.5]
)

# Compare results with the known population mean
true_mean = after["satisfaction"].mean()
observed_mean = after_observed.mean()

print("Observed post-redesign mean:", observed_mean)
print("Revised weighted mean:", weighted_mean_v2)
print("True population mean:", true_mean)
print("Remaining bias:", weighted_mean_v2 - true_mean)
print("Revised 95% bootstrap confidence interval:", weighted_ci_v2)

2. What changed from Version 1?
Version 1 resampled only the respondents and used their previously calculated weights.

Version 2 makes three important changes:
1. Resamples the full population: Each bootstrap sample includes both respondents and nonrespondents, allowing the response model to be re-estimated.
2. Refits the logistic regression: Each repetition estimates a new relationship between the observed covariates and survey participation.
3. Recalculates the weights: The updated response probabilities are used to calculate new inverse-probability weights before estimating satisfaction.
These changes allow the bootstrap to account for variation in both the observed survey responses and the estimated response probabilities.

The original weighted point estimate should remain approximately 4.1432 because we have not changed the underlying weighting model. However, the revised confidence interval may differ because we now refit the model in each bootstrap sample.
3. Mechanism check: What assumptions are required?
For inverse-probability weighting to recover the true population mean, several assumptions must hold.

Conditional independence: After controlling for engagement_minutes and had_support_ticket, whether someone responds must be independent of their satisfaction score.

Mathematically:
\[
R \perp S \mid X
\]

Here, \(R\) represents survey participation, \(S\) represents satisfaction, and \(X\) represents the observed covariates.

Positivity: Every type of user must have a positive probability of responding. Otherwise, users who never respond cannot be represented through weighting.

Correct response model: The logistic regression must adequately estimate the probability of responding based on the available covariates.

Our simulated survey violates the conditional independence assumption because the probability of responding depends directly on satisfaction.

For example, two users with identical engagement minutes and support-ticket histories may still have different probabilities of responding if one is more satisfied than the other.

As a result, weighting based only on the two observed covariates cannot fully account for the selection mechanism.
4. Evaluation: Did Version 2 correct the bias?
Based on Version 1, our results were:
Measure	Version 1
Observed mean	4.3041
Weighted mean	4.1432
True population mean	3.3860
Remaining bias	+0.7572
95% bootstrap confidence interval	[4.1224, 4.1634]
Because Version 2 uses the same logistic regression model for the original point estimate, we expect the weighted mean to remain approximately 4.1432, with a remaining bias of approximately 0.7572 points.

The updated 95% confidence interval must be calculated by running the revised code. Its exact endpoints cannot be determined from the Version 1 results alone.

Even with the improved bootstrap, the weighted estimate remains substantially higher than the true population mean of 3.38602.

Therefore, Version 2 improves how we estimate uncertainty, but it does not correct the underlying nonresponse bias.

The problem is not simply that our confidence interval was calculated incorrectly. The larger issue is that users with higher satisfaction scores are more likely to respond, even after controlling for the two observed covariates.

Refitting the logistic regression during each bootstrap repetition does not change that selection mechanism.
5. Final conclusion
Version 2 addresses a meaningful limitation in the original bootstrap procedure by recalculating response probabilities and weights in every resample. This provides a more complete assessment of uncertainty associated with the weighting method.

However, the correction still relies on the assumption that survey participation is independent of satisfaction after controlling for the observed covariates. Since our simulated response mechanism violates this assumption, the weighted estimate remains biased.

The main takeaway is that improving statistical precision does not necessarily improve accuracy. A better bootstrap can help us understand uncertainty around an estimate, but it cannot fix a response mechanism that systematically overrepresents satisfied users.

To properly address this problem, the company would need to improve how survey responses are collected, ideally by randomly selecting users and reducing or accounting for nonresponse rather than relying only on users who voluntarily complete the existing survey.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.